# Lab 3: Geoparsing II

In this lab session, we'll:
- Evaluate how well spaCy's trained pipelines perform geoparsing
- Learn how to compute evaluation metrics: Precision, Recall, and F1 score
- Observe micro-level successes and failures, i.e., individual correct and incorrect predictions
- Create a ground-truth dataset for evaluation

```text
Text           "We flew from London to Paris, then drove to Lake Geneva."
                                        │
                                        ▼
Ground truth   London   Paris   Lake Geneva
               (place names annotated by humans)
                                        │
                                        ▼
Prediction     London   Paris
               (place names found by spaCy)
                                        │
                                        ▼
Evaluation     Precision = 2/2 = 1.0   Recall = 2/3 ≈ 0.67
               (compare the prediction with the ground truth)
```

Below I will provide code along with explanations.

## Part 1: Creating a Ground-Truth Dataset

To evaluate a geoparser, we need to know the correct answers first. A **ground-truth dataset** is a set of texts in which humans have annotated all place names. Later, we'll compare spaCy's predictions with these annotations.

We'll store the dataset in **JSON** (JavaScript Object Notation) format. Why JSON? (Learn more at [json.org](https://www.json.org/json-en.html).)
- **Readable**: both humans and computers can read it easily
- **Structured**: each text can be stored together with its place names
- **Common**: almost every programming language and tool can read JSON

In Python, a JSON dataset looks like a list of dictionaries. Each dictionary is one sample with two keys:
- `"text"`: the text itself
- `"places"`: the place names in the text, annotated by us

```text
dataset  (a list: [ ... ])
│
├── sample 1  (a dictionary: { ... })
│   ├── "text":   "Jordan flew from London to Paris for a conference."
│   └── "places": ["London", "Paris"]
│
├── sample 2
│   ├── "text":   "Last summer, we went swimming in Lake Geneva."
│   └── "places": ["Lake Geneva"]
│
└── ...
```

In [ ]:
# Our ground-truth dataset: a list of samples
dataset = [
    {
        "text": "Jordan flew from London to Paris for a conference.",
        "places": ["London", "Paris"]
    },
    {
        "text": "Last summer, we went swimming in Lake Geneva.",
        "places": ["Lake Geneva"]
    },
    {
        "text": "Washington moved to Seattle after graduation.",
        "places": ["Seattle"]
    },
]

# Print each sample
for sample in dataset:
    print(sample["text"])
    print(sample["places"])
    print()

Next, we save the dataset as a JSON file, so that we can reuse it later or share it with others.

To do this, we use Python's built-in `json` package and call `json.dump()` with:
- `dataset`: the data to save
- `f`: the file to write into
- `indent=4`: adds line breaks and spaces, so the file is easy to read

In [ ]:
import json

# Save the dataset to a JSON file
with open("ground_truth.json", "w") as f:
    json.dump(dataset, f, indent=4)

Let's check that the file was saved correctly. We load it back with `json.load()`.

In [ ]:
# Load the dataset from the JSON file
with open("ground_truth.json", "r") as f:
    loaded_dataset = json.load(f)

print(loaded_dataset)

### Exercise 1

Now it's your turn to **create your own ground-truth dataset**:
1. Collect 5 texts from any source you like, e.g., news articles, Wikipedia, travel blogs, or social media posts.
2. For each text, annotate all place names by hand.
3. Store your samples in the same JSON format as above and save them as `my_ground_truth.json`.

Tip: Texts with tricky cases make a more interesting dataset, e.g., a person named after a place, or a place name with several words.

In [ ]:
# Create your own ground-truth dataset below
my_dataset = [
    {
        "text": "...",
        "places": [...]
    },
]

# Save your dataset to a JSON file
with open("my_ground_truth.json", "w") as f:
    json.dump(my_dataset, f, indent=4)

## Part 2: Computing Evaluation Metrics

Now we evaluate spaCy on our ground-truth dataset. First, we need spaCy's predictions. As in Lab 2, we load the small English pipeline and keep only the entities whose label can refer to a place.

To get the predictions, we write a **function**. A function is a named block of code that we can reuse: we give it an input, it runs the code inside, and it gives back an output.

```text
Input             +--------------------------+         Output
"Jordan flew  --> |  predict_places(text)    | --> ["London", "Paris"]
 from London      |  1. run spaCy on text    |
 to Paris ..."    |  2. keep place entities  |
                  +--------------------------+
```

In Python, a function looks like this:
- `def`: starts the function, followed by its name and its input in brackets
- the indented lines: the code inside the function
- `return`: gives back the output

Once defined, we can call it as many times as we like, e.g., `predict_places("We took a boat trip on the Danube.")`.

Inside the function, we use a short way to build a list, called a **list comprehension**:

```python
[ent.text for ent in doc.ents if ent.label_ in place_labels]
```

Read it from left to right:
- `ent.text`: what to put in the list
- `for ent in doc.ents`: go through each entity
- `if ent.label_ in place_labels`: keep only the place entities

It does the same as this loop from Lab 2, but puts the results into a list instead of printing them:

```python
places = []
for ent in doc.ents:
    if ent.label_ in place_labels:
        places.append(ent.text)
```

In [ ]:
import spacy

# Load the small English pipeline
nlp = spacy.load("en_core_web_sm")

# Labels that can refer to a place
place_labels = ["GPE", "LOC", "FAC", "ORG", "NORP"]

def predict_places(text):
    """
    Return the place names that spaCy predicts in a text
    """
    doc = nlp(text)
    return [ent.text for ent in doc.ents if ent.label_ in place_labels]

# Compare the ground truth and the prediction for each sample
for sample in dataset:
    print("Text:        ", sample["text"])
    print("Ground truth:", sample["places"])
    print("Prediction:  ", predict_places(sample["text"]))
    print()

Next, we count the three cases from the lecture by comparing the two lists:
- **TP** (true positives): predicted, and also in the ground truth
- **FP** (false positives): predicted, but not in the ground truth
- **FN** (false negatives): in the ground truth, but not predicted

Then we compute the metrics:
- **Precision** = TP / (TP + FP)
- **Recall** = TP / (TP + FN)
- **F1** = 2 / (1/Precision + 1/Recall)

In the code, we count each case with the same kind of list comprehension, e.g.:

```python
len([p for p in prediction if p in truth])
```

- `[p for p in prediction if p in truth]`: the predicted place names that are also in the ground truth
- `len(...)`: how many there are, i.e., the TP count for one sample
- `tp += ...`: adds this count to the total over all samples

In [ ]:
def evaluate(dataset):
    """
    Compute Precision, Recall, and F1 score of spaCy on a dataset
    """
    tp, fp, fn = 0, 0, 0

    # Count TP, FP, and FN over all samples
    for sample in dataset:
        truth = sample["places"]
        prediction = predict_places(sample["text"])

        tp += len([p for p in prediction if p in truth])
        fp += len([p for p in prediction if p not in truth])
        fn += len([t for t in truth if t not in prediction])

    # Compute the metrics
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 / (1 / precision + 1 / recall)

    print("TP:", tp, " FP:", fp, " FN:", fn)
    print("Precision:", round(precision, 2))
    print("Recall:   ", round(recall, 2))
    print("F1 score: ", round(f1, 2))

# Evaluate spaCy on our dataset
evaluate(dataset)

### Exercise 2

Now evaluate spaCy on **your own dataset** from Exercise 1. Load `my_ground_truth.json` and pass it to `evaluate()`.

In [ ]:
# Load your own dataset
with open("my_ground_truth.json", "r") as f:
    my_dataset = json.load(f)

# Evaluate spaCy on your dataset
evaluate(my_dataset)

## Part 3: Observing Successes and Failures

Evaluation metrics give us one number for a whole dataset. But they don't tell us **why** a geoparser succeeds or fails. In this part, we look at individual predictions, i.e., **micro-level** evaluation.

We'll study three factors from the lecture:
- **Input context**: Does the text around a place name matter?
- **Language data and pipeline**: What happens when the text and the pipeline are in different languages?
- **Model size**: Does a larger pipeline do better?

We'll compare three pipelines. You already know the first two from Lab 2:
- [`en_core_web_sm`](https://spacy.io/models/en#en_core_web_sm): the small English pipeline
- [`de_core_news_sm`](https://spacy.io/models/de#de_core_news_sm): the small German pipeline
- [`en_core_web_lg`](https://spacy.io/models/en#en_core_web_lg): the **large** English pipeline (about 400 MB). Check its [model page](https://spacy.io/models/en#en_core_web_lg) to see how its size and accuracy compare with `en_core_web_sm`

In [ ]:
from spacy import displacy

# Load the three pipelines
nlp_en_sm = spacy.load("en_core_web_sm")
nlp_de_sm = spacy.load("de_core_news_sm")
nlp_en_lg = spacy.load("en_core_web_lg")

For each sample, we want to see three things:
- **Correct**: place names found by spaCy (TP)
- **Wrong**: other words that spaCy took as place names (FP)
- **Missed**: place names that spaCy didn't find (FN)

The function below prints these lists and shows the displaCy visualization, so we can also see which label spaCy gave to each entity. What does each label mean? Check the table of entity types in the [spaCy documentation](https://v2.spacy.io/api/annotation#named-entities).

In [ ]:
def observe(sample, pipeline):
    """
    Show the correct, wrong, and missed place names for one sample
    """
    doc = pipeline(sample["text"])
    truth = sample["places"]
    prediction = [ent.text for ent in doc.ents if ent.label_ in place_labels]

    print("Correct:", [p for p in prediction if p in truth])
    print("Wrong:  ", [p for p in prediction if p not in truth])
    print("Missed: ", [t for t in truth if t not in prediction])
    displacy.render(doc, style="ent")

### 3.1 Input Context

We give the small English pipeline the same place name twice: once on its own, and once inside a sentence.

In [ ]:
context_dataset = [
    {"text": "Kärntner Straße", "places": ["Kärntner Straße"]},
    {"text": "The first Christmas lights were installed on Kärntner Straße.", "places": ["Kärntner Straße"]},
    {"text": "Danube", "places": ["Danube"]},
    {"text": "We took a boat trip on the Danube.", "places": ["Danube"]},
    {"text": "Graben", "places": ["Graben"]},
    {"text": "We went shopping on the Graben in Vienna.", "places": ["Graben", "Vienna"]},
]

for sample in context_dataset:
    observe(sample, nlp_en_sm)
    print()

**Question 1**: Compare each place name on its own with the same place name in a sentence. Does the context change the result? Give one example.

**Answer**: ...

### 3.2 Language Data and Pipeline

Now we have the same sentences in German and English. We run **both** the English and the German pipeline on **all** of them.

In [ ]:
language_dataset = [
    {"text": "In der Kärntner Straße wurden die ersten Weihnachtsbeleuchtungen montiert.", "places": ["Kärntner Straße"]},
    {"text": "The first Christmas lights were installed on Kärntner Straße.", "places": ["Kärntner Straße"]},
    {"text": "Wir sind mit dem Zug von Wien nach Salzburg gefahren.", "places": ["Wien", "Salzburg"]},
    {"text": "We took the train from Vienna to Salzburg.", "places": ["Vienna", "Salzburg"]},
]

for sample in language_dataset:
    print("--- English pipeline ---")
    observe(sample, nlp_en_sm)
    print("--- German pipeline ---")
    observe(sample, nlp_de_sm)
    print()

**Question 2**: Which pipeline works better when its language matches the text? What goes wrong when it doesn't? Give one example.

**Answer**: ...

### 3.3 Model Size

Finally, we run the small and the large English pipeline on the same English sentences.

In [ ]:
size_dataset = [
    {"text": "The first Christmas lights were installed on Kärntner Straße.", "places": ["Kärntner Straße"]},
    {"text": "Jordan flew from London to Paris for a conference.", "places": ["London", "Paris"]},
    {"text": "We went shopping on the Graben in Vienna.", "places": ["Graben", "Vienna"]},
    {"text": "We took a boat trip on the Danube.", "places": ["Danube"]},
]

for sample in size_dataset:
    print("--- Small pipeline ---")
    observe(sample, nlp_en_sm)
    print("--- Large pipeline ---")
    observe(sample, nlp_en_lg)
    print()

**Question 3**: Does the large pipeline always do better than the small one? Give one example.

**Answer**: ...

### Exercise 3

Now it's your turn to **create a second dataset** to study successes and failures:
1. Write at least 2 samples for each factor: input context, language, and model size.
2. Annotate the place names by hand and save the dataset as `my_micro_dataset.json`.
3. Run `observe()` with the pipelines you need.

Tip: Try place names that are rare, have several words, or also work as a person's name.

In [ ]:
# Create your second dataset below
my_micro_dataset = [
    {"text": "...", "places": [...]},
]

# Save your dataset to a JSON file
with open("my_micro_dataset.json", "w") as f:
    json.dump(my_micro_dataset, f, indent=4)

# Observe the results

**Question 4**: Describe one success and one failure from your own dataset. Why do you think spaCy behaved that way?

**Answer**: ...